# Notebook 2: Validación de Inputs y Seguridad Básica

## Objetivos
- Implementar validación segura de inputs en agentes de IA
- Desarrollar sanitización de entradas de usuario
- Crear mecanismos de validación de outputs
- Establecer controles de acceso básicos
- Aplicar principios de seguridad al agente existente

## Configuración del Agente

In [ ]:
import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_base=os.environ.get("GITHUB_BASE_URL"),
        openai_api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langsmith import Client

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

client = Client(None)
prompt = client.pull_prompt("hwchase17/openai-tools-agent")

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")

## 1. Evaluación Segura de Expresiones

Implementamos una función de evaluación segura que solo permite operaciones matemáticas básicas:

In [ ]:
def safe_eval(expression):
    """
    Evalúa solo expresiones matemáticas seguras.
    
    Args:
        expression: String con expresión matemática
    
    Returns:
        Resultado de la evaluación o mensaje de error
    """
    # Caracteres permitidos: números y operadores matemáticos básicos
    allowed = set('0123456789+-*/(). ')
    
    # Verificar que solo contiene caracteres permitidos
    if not set(expression) <= allowed:
        return "❌ Expresión no permitida: contiene caracteres inválidos."
    
    try:
        result = eval(expression)
        return f"✅ Resultado: {result}"
    except Exception as e:
        return f"❌ Error en la expresión: {e}"

# Pruebas de safe_eval
test_expressions = [
    "2 + 2",
    "10 * (5 + 3)",
    "__import__('os').system('ls')",  # Intento de inyección de código
    "eval('print(1)')",  # Intento de ejecución de código
    "100 / 0",  # Error matemático
    "import os; os.system('rm -rf /')",  # Comando peligroso
]

print("🧪 Pruebas de Evaluación Segura:")
for expr in test_expressions:
    print(f"\nExpresión: {expr}")
    print(f"Resultado: {safe_eval(expr)}")

## 2. Sanitización de Inputs

Implementamos un sistema completo de sanitización de inputs:

In [ ]:
import re

class InputSanitizer:
    """
    Clase para sanitizar inputs de usuario en agentes de IA.
    """
    
    def __init__(self, max_length=1000):
        self.max_length = max_length
        self.dangerous_chars = r'[<>"\';&|`$]'
        self.dangerous_keywords = [
            'import', 'exec', 'eval', 'compile',
            '__import__', 'open', 'file', 'os.system',
            'subprocess', 'pickle', 'marshal'
        ]
    
    def sanitize_input(self, user_input):
        """
        Sanitiza el input del usuario removiendo caracteres peligrosos.
        
        Args:
            user_input: String de entrada del usuario
        
        Returns:
            String sanitizado
        """
        # Remover caracteres peligrosos
        cleaned = re.sub(self.dangerous_chars, '', user_input)
        
        # Limitar longitud
        if len(cleaned) > self.max_length:
            cleaned = cleaned[:self.max_length]
            print(f"⚠️ Input truncado a {self.max_length} caracteres")
        
        return cleaned
    
    def check_dangerous_keywords(self, user_input):
        """
        Verifica si el input contiene palabras clave peligrosas.
        
        Args:
            user_input: String de entrada del usuario
        
        Returns:
            Tuple (bool, list) - (es_peligroso, palabras_encontradas)
        """
        input_lower = user_input.lower()
        found_keywords = []
        
        for keyword in self.dangerous_keywords:
            if keyword in input_lower:
                found_keywords.append(keyword)
        
        return len(found_keywords) > 0, found_keywords
    
    def validate_input(self, user_input):
        """
        Validación completa del input.
        
        Args:
            user_input: String de entrada del usuario
        
        Returns:
            Tuple (bool, str, str) - (es_valido, mensaje, input_sanitizado)
        """
        # Verificar palabras clave peligrosas
        is_dangerous, keywords = self.check_dangerous_keywords(user_input)
        if is_dangerous:
            return False, f"❌ Input contiene palabras clave peligrosas: {keywords}", None
        
        # Sanitizar input
        sanitized = self.sanitize_input(user_input)
        
        # Verificar si el input fue modificado significativamente
        if len(sanitized) < len(user_input) * 0.8:
            return False, "❌ Input contiene demasiados caracteres peligrosos", None
        
        return True, "✅ Input validado correctamente", sanitized

# Pruebas del InputSanitizer
sanitizer = InputSanitizer(max_length=1000)

test_inputs = [
    "¿Qué es la inteligencia artificial?",
    "<script>alert('xss')</script>",
    "import os; os.system('ls')",
    "exec('print(1)')",
    "SELECT * FROM users WHERE 1=1",
    "a" * 1500,  # Input muy largo
    "Información normal sobre Python"
]

print("🧪 Pruebas de Sanitización de Inputs:")
for test_input in test_inputs:
    print(f"\nInput original: {test_input[:50]}..." if len(test_input) > 50 else f"\nInput original: {test_input}")
    is_valid, message, sanitized = sanitizer.validate_input(test_input)
    print(f"{message}")
    if sanitized:
        print(f"Input sanitizado: {sanitized[:50]}..." if len(sanitized) > 50 else f"Input sanitizado: {sanitized}")

## 3. Validación de Outputs

Implementamos validación de respuestas para evitar filtración de información sensible:

In [ ]:
class OutputValidator:
    """
    Clase para validar outputs del agente de IA.
    """
    
    def __init__(self):
        self.dangerous_patterns = [
            r'password:\s*\w+',
            r'api[_-]?key:\s*\w+',
            r'token:\s*\w+',
            r'secret:\s*\w+',
            r'credential:\s*\w+',
            r'eval\s*\(',
            r'exec\s*\(',
            r'__import__\s*\(',
            r'\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b',  # Credit cards
            r'\b\d{3}-\d{2}-\d{4}\b',  # SSN
            r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'  # Email
        ]
    
    def validate_response(self, response):
        """
        Valida que la respuesta no contenga patrones peligrosos.
        
        Args:
            response: String de respuesta del agente
        
        Returns:
            Tuple (bool, str) - (es_valido, mensaje)
        """
        for pattern in self.dangerous_patterns:
            if re.search(pattern, response, re.IGNORECASE):
                return False, f"❌ Response bloqueada: contiene patrón peligroso '{pattern}'"
        
        return True, "✅ Response validada correctamente"
    
    def sanitize_response(self, response):
        """
        Sanitiza la respuesta removiendo información sensible.
        
        Args:
            response: String de respuesta del agente
        
        Returns:
            String sanitizado
        """
        sanitized = response
        
        # Remover patrones de información sensible
        sensitive_patterns = [
            (r'\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b', '[CARD_NUMBER_REDACTED]'),
            (r'\b\d{3}-\d{2}-\d{4}\b', '[SSN_REDACTED]'),
            (r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '[EMAIL_REDACTED]'),
            (r'password:\s*\w+', 'password: [REDACTED]'),
            (r'api[_-]?key:\s*\w+', 'api_key: [REDACTED]'),
            (r'token:\s*\w+', 'token: [REDACTED]')
        ]
        
        for pattern, replacement in sensitive_patterns:
            sanitized = re.sub(pattern, replacement, sanitized, flags=re.IGNORECASE)
        
        return sanitized

# Pruebas del OutputValidator
output_validator = OutputValidator()

test_responses = [
    "La inteligencia artificial es una rama de la informática.",
    "Tu password es: secreto123",
    "La api_key para acceder es: sk-1234567890",
    "El token de sesión es: abc123def456",
    "Puedes usar eval(print('hola')) para ejecutar código",
    "Contacta a usuario@ejemplo.com para más información",
    "El número de tarjeta es 1234-5678-9012-3456"
]

print("🧪 Pruebas de Validación de Outputs:")
for response in test_responses:
    print(f"\nResponse original: {response}")
    is_valid, message = output_validator.validate_response(response)
    print(f"{message}")
    if not is_valid:
        sanitized = output_validator.sanitize_response(response)
        print(f"Response sanitizada: {sanitized}")

## 4. Sistema de Control de Acceso

Implementamos un sistema básico de control de acceso:

In [ ]:
from collections import defaultdict
import time
import hashlib

class AccessControl:
    """
    Sistema de control de acceso para agentes de IA.
    """
    
    def __init__(self, requests_per_minute=60):
        self.requests_per_minute = requests_per_minute
        self.user_requests = defaultdict(list)
        self.user_permissions = defaultdict(set)
        self.blocked_users = set()
    
    def is_allowed(self, user_id):
        """
        Verifica si el usuario tiene permitido hacer una request.
        
        Args:
            user_id: Identificador del usuario
        
        Returns:
            bool - True si está permitido, False si no
        """
        # Verificar si el usuario está bloqueado
        if user_id in self.blocked_users:
            return False
        
        now = time.time()
        minute_ago = now - 60
        
        # Limpiar requests antiguas
        self.user_requests[user_id] = [
            req_time for req_time in self.user_requests[user_id] 
            if req_time > minute_ago
        ]
        
        # Verificar límite
        if len(self.user_requests[user_id]) >= self.requests_per_minute:
            return False
        
        # Registrar request
        self.user_requests[user_id].append(now)
        return True
    
    def grant_permission(self, user_id, permission):
        """
        Otorga un permiso específico a un usuario.
        
        Args:
            user_id: Identificador del usuario
            permission: Permiso a otorgar
        """
        self.user_permissions[user_id].add(permission)
    
    def check_permission(self, user_id, permission):
        """
        Verifica si un usuario tiene un permiso específico.
        
        Args:
            user_id: Identificador del usuario
            permission: Permiso a verificar
        
        Returns:
            bool - True si tiene el permiso, False si no
        """
        return permission in self.user_permissions[user_id]
    
    def block_user(self, user_id):
        """
        Bloquea a un usuario.
        
        Args:
            user_id: Identificador del usuario
        """
        self.blocked_users.add(user_id)
    
    def unblock_user(self, user_id):
        """
        Desbloquea a un usuario.
        
        Args:
            user_id: Identificador del usuario
        """
        self.blocked_users.discard(user_id)
    
    def get_user_stats(self, user_id):
        """
        Obtiene estadísticas de uso de un usuario.
        
        Args:
            user_id: Identificador del usuario
        
        Returns:
            dict con estadísticas
        """
        now = time.time()
        minute_ago = now - 60
        
        recent_requests = [
            req_time for req_time in self.user_requests[user_id] 
            if req_time > minute_ago
        ]
        
        return {
            "requests_last_minute": len(recent_requests),
            "permissions": list(self.user_permissions[user_id]),
            "is_blocked": user_id in self.blocked_users
        }

# Pruebas del AccessControl
access_control = AccessControl(requests_per_minute=5)

# Configurar permisos
access_control.grant_permission("user1", "read")
access_control.grant_permission("user1", "write")
access_control.grant_permission("user2", "read")

# Bloquear un usuario
access_control.block_user("user3")

print("🧪 Pruebas de Control de Acceso:")

# Test rate limiting
print("\n--- Test Rate Limiting ---")
for i in range(7):
    allowed = access_control.is_allowed("user1")
    print(f"Request {i+1}: {'✅ Permitido' if allowed else '❌ Bloqueado por rate limit'}")

# Test permisos
print("\n--- Test Permisos ---")
print(f"user1 tiene permiso 'read': {access_control.check_permission('user1', 'read')}")
print(f"user1 tiene permiso 'admin': {access_control.check_permission('user1', 'admin')}")
print(f"user2 tiene permiso 'read': {access_control.check_permission('user2', 'read')}")

# Test usuarios bloqueados
print("\n--- Test Usuarios Bloqueados ---")
print(f"user3 puede hacer requests: {access_control.is_allowed('user3')}")

# Test estadísticas
print("\n--- Test Estadísticas ---")
print(f"Estadísticas user1: {access_control.get_user_stats('user1')}")

## 5. Agente Seguro - Integración de Componentes

Integramos todos los componentes de seguridad en un wrapper para el agente:

In [ ]:
class SecureAgentWrapper:
    """
    Wrapper que añade capas de seguridad al agente de IA.
    """
    
    def __init__(self, agent_executor):
        self.agent_executor = agent_executor
        self.input_sanitizer = InputSanitizer(max_length=1000)
        self.output_validator = OutputValidator()
        self.access_control = AccessControl(requests_per_minute=60)
        self.audit_log = []
    
    def invoke(self, user_id, user_input):
        """
        Invoca el agente con todas las capas de seguridad.
        
        Args:
            user_id: Identificador del usuario
            user_input: Input del usuario
        
        Returns:
            dict con respuesta y metadatos
        """
        # 1. Verificar control de acceso
        if not self.access_control.is_allowed(user_id):
            self._log_request(user_id, user_input, "blocked", "Rate limit exceeded")
            return {
                "success": False,
                "message": "❌ Request bloqueada: límite de rate excedido",
                "response": None
            }
        
        # 2. Validar input
        is_valid, message, sanitized_input = self.input_sanitizer.validate_input(user_input)
        if not is_valid:
            self._log_request(user_id, user_input, "blocked", message)
            return {
                "success": False,
                "message": message,
                "response": None
            }
        
        # 3. Invocar agente
        try:
            response = self.agent_executor.invoke({"input": sanitized_input})
            raw_output = response['output']
        
            # 4. Validar output
            is_valid, message = self.output_validator.validate_response(raw_output)
            if not is_valid:
                sanitized_output = self.output_validator.sanitize_response(raw_output)
                self._log_request(user_id, user_input, "sanitized", message)
                return {
                    "success": True,
                    "message": "⚠️ Response sanitizada por seguridad",
                    "response": sanitized_output
                }
            
            self._log_request(user_id, user_input, "success", "Request completada")
            return {
                "success": True,
                "message": "✅ Request completada exitosamente",
                "response": raw_output
            }
            
        except Exception as e:
            self._log_request(user_id, user_input, "error", str(e))
            return {
                "success": False,
                "message": f"❌ Error en el agente: {e}",
                "response": None
            }
    
    def _log_request(self, user_id, input_text, status, message):
        """
        Registra la request en el audit log.
        """
        log_entry = {
            "timestamp": time.time(),
            "user_id": user_id,
            "input": input_text[:100],  # Solo primeros 100 caracteres
            "status": status,
            "message": message
        }
        self.audit_log.append(log_entry)
    
    def get_audit_log(self, user_id=None):
        """
        Obtiene el audit log, opcionalmente filtrado por usuario.
        
        Args:
            user_id: Identificador del usuario (opcional)
        
        Returns:
            list de entradas de log
        """
        if user_id:
            return [entry for entry in self.audit_log if entry['user_id'] == user_id]
        return self.audit_log

# Crear el agente seguro
secure_agent = SecureAgentWrapper(agent_executor)

print("✅ Agente seguro configurado con todas las capas de protección.")

## 6. Práctica con el Agente Seguro

Probamos el agente seguro con diferentes tipos de inputs:

In [ ]:
# Pruebas del agente seguro
test_scenarios = [
    {
        "user_id": "user_test_1",
        "input": "¿Qué es la inteligencia artificial?",
        "description": "Consulta normal"
    },
    {
        "user_id": "user_test_2",
        "input": "<script>alert('xss')</script> ¿Qué es Python?",
        "description": "Input con caracteres peligrosos"
    },
    {
        "user_id": "user_test_3",
        "input": "import os; os.system('ls')",
        "description": "Intento de inyección de código"
    },
    {
        "user_id": "user_test_4",
        "input": "¿Cuáles son los principios de la ética en IA?",
        "description": "Consulta sobre ética"
    }
]

print("🧪 Pruebas del Agente Seguro:")
for scenario in test_scenarios:
    print(f"\n--- {scenario['description']} ---")
    print(f"User ID: {scenario['user_id']}")
    print(f"Input: {scenario['input']}")
    
    result = secure_agent.invoke(scenario['user_id'], scenario['input'])
    print(f"\n{result['message']}")
    if result['response']:
        print(f"Response: {result['response'][:200]}..." if len(result['response']) > 200 else f"Response: {result['response']}")

## 7. Verificación del Audit Log

Revisamos el audit log para monitorear las requests:

In [ ]:
# Obtener y mostrar el audit log
audit_log = secure_agent.get_audit_log()

print("📋 Audit Log del Agente Seguro:")
print(f"Total de requests: {len(audit_log)}")

for entry in audit_log:
    print(f"\n--- Entry ---")
    print(f"Timestamp: {entry['timestamp']}")
    print(f"User ID: {entry['user_id']}")
    print(f"Input: {entry['input']}")
    print(f"Status: {entry['status']}")
    print(f"Message: {entry['message']}")

## 8. Principios de Seguridad Implementados

Resumen de los principios de seguridad implementados:

In [ ]:
# Resumen de principios implementados
security_principles = {
    "Input Validation": {
        "implementado": True,
        "componentes": [
            "Sanitización de caracteres peligrosos",
            "Detección de palabras clave maliciosas",
            "Limitación de longitud",
            "Validación de integridad"
        ]
    },
    "Output Validation": {
        "implementado": True,
        "componentes": [
            "Detección de información sensible",
            "Sanitización de datos confidenciales",
            "Validación de patrones peligrosos"
        ]
    },
    "Access Control": {
        "implementado": True,
        "componentes": [
            "Rate limiting",
            "Sistema de permisos",
            "Bloqueo de usuarios",
            "Estadísticas de uso"
        ]
    },
    "Audit Logging": {
        "implementado": True,
        "componentes": [
            "Registro de todas las requests",
            "Tracking de status",
            "Mensajes de error",
            "Filtrado por usuario"
        ]
    },
    "Safe Execution": {
        "implementado": True,
        "componentes": [
            "Evaluación segura de expresiones",
            "Manejo de errores controlado",
            "Prevención de inyección de código"
        ]
    }
}

print("🏗️ Principios de Seguridad Implementados:")
for principle, info in security_principles.items():
    status = "✅" if info['implementado'] else "❌"
    print(f"\n{status} {principle}:")
    for component in info['componentes']:
        print(f"  • {component}")

## 9. Resumen

### Componentes Implementados
- **InputSanitizer**: Sanitización y validación de inputs de usuario
- **OutputValidator**: Validación y sanitización de outputs del agente
- **AccessControl**: Sistema de control de acceso y rate limiting
- **SecureAgentWrapper**: Integración de todas las capas de seguridad
- **Audit Logging**: Registro de todas las requests para monitoreo

### Principios de Seguridad Aplicados
- **Input validation**: Validación exhaustiva de entradas
- **Output validation**: Protección contra filtración de datos
- **Access control**: Principio de menor privilegio
- **Rate limiting**: Protección contra abuso de recursos
- **Audit logging**: Trazabilidad de todas las operaciones

### Próximos Pasos
- Notebook 3: Implementación de frameworks éticos y filtros de contenido
- Notebook 4: Protección contra ataques específicos (prompt injection, adversarial attacks)
- Notebook 5: Governance, compliance y monitoring avanzado

### Mejoras Futuras
- Implementar autenticación de usuarios
- Agregar más patrones de detección de amenazas
- Implementar machine learning para detección de anomalías
- Agregar cifrado de datos sensibles en el audit log